# Entrega sprint semana 6 — cierre de pendientes

Este notebook complementa el avance `sprint2_fe_ab_testing(2).ipynb` y cubre los puntos que exige la foto de la entrega: validacion sin leakage, tabla final comparable, grafico unico, reproducibilidad, decision tecnica, riesgos y siguientes pasos.

> Uso recomendado: ejecutar primero el notebook avanzado hasta crear `df_eval`, `tabla_resultados`, `pred_a`, `pred_b`, `pred_c`, `pred_d` y luego ejecutar este notebook.


## 1. Que estaba pendiente respecto a la guia

| Punto de la guia | Estado en el avance | Pendiente que cubre este notebook |
|---|---|---|
| Contexto, objetivo, dataset y metrica central | Ya esta | Se resume para exposicion |
| Baseline y metrica | Ya esta | Se deja recordatorio breve |
| Experimentos A/B | Ya esta | Se homologa a tabla final |
| Resultados comparables | Parcial | Tabla final con metrica principal, secundarias y latencia |
| Grafico unico | Parcial | Grafico unico de PR curve + ranking |
| Validacion sin leakage | Parcial/inconsistente | Split temporal o grupal y transformaciones ajustadas solo con train |
| Reproducibilidad | Parcial | README, logs, snapshot y hashes |
| Decision tecnica | Parcial | Decision automatica segun metrica |
| Riesgos y proximos pasos | Falta | Se agrega cierre ejecutivo |


In [ ]:
# ============================================================================
# 0. CONFIGURACION GENERAL
# ============================================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    precision_recall_curve
)
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')

SEED = globals().get('SEED', 42)
CONTAM = globals().get('CONTAM', 0.05)
COL_GRUPO = globals().get('COL_R', 'NUM_SPN_R')
COL_VU = 'VU'

OUTPUT_DIR = Path('outputs_semana6')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICAS_PATH = OUTPUT_DIR / 'metrics_semana6.csv'
README_PATH = OUTPUT_DIR / 'README_REPRODUCIBILIDAD.txt'
SNAPSHOT_PATH = OUTPUT_DIR / 'snapshot_semana6.json'
FIG_PATH = OUTPUT_DIR / 'fig_semana6_resultado_unico.png'

print('Configuracion lista')
print(f'SEED        : {SEED}')
print(f'CONTAM      : {CONTAM}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR.resolve()}')


## 2. Validacion de insumos

Este bloque verifica que el notebook anterior haya dejado los objetos minimos para cerrar la entrega.


In [ ]:
# ============================================================================
# 1. VALIDACION DE INSUMOS
# ============================================================================

def validar_insumos(df: pd.DataFrame) -> None:
    """
    Valida que el DataFrame tenga las columnas minimas para ejecutar la entrega.

    Args:
        df (pd.DataFrame): Base de evaluacion generada en el notebook anterior.

    Returns:
        None. Lanza ValueError si faltan columnas obligatorias.
    """
    columnas_requeridas = [
        'NUM_SPN_R',
        'FOB_DOLAR',
        'VU',
        'VU_LOG',
        'VU_WINS',
        '_gt'
    ]

    faltantes = [
        columna
        for columna in columnas_requeridas
        if columna not in df.columns
    ]

    if faltantes:
        raise ValueError(f'Columnas faltantes en df_eval: {faltantes}')

    if df.empty:
        raise ValueError('df_eval esta vacio.')

    if df['_gt'].nunique() < 2:
        raise ValueError('La etiqueta proxy _gt debe tener clase normal y clase outlier.')


if 'df_eval' not in globals():
    raise NameError(
        'No existe df_eval. Ejecuta primero sprint2_fe_ab_testing(2).ipynb '
        'hasta la seccion de construccion de df_eval.'
    )

df_eval = df_eval.copy().reset_index(drop=True)
validar_insumos(df_eval)

print('Insumos validados')
print(f'Registros          : {len(df_eval):,}')
print(f'Subpartidas        : {df_eval["NUM_SPN_R"].nunique():,}')
print(f'Outliers proxy     : {df_eval["_gt"].sum():,}')
print(f'Tasa outlier proxy : {100 * df_eval["_gt"].mean():.2f}%')


## 3. Split correcto y control de leakage

La guia pide split correcto y sin leakage. Aqui se aplica una regla practica:

1. Si existe columna temporal, se usa split temporal: train con fechas antiguas y test con fechas recientes.
2. Si no existe columna temporal util, se usa split grupal por aduana cuando existe `ADUANA`.
3. Si no existe aduana, se usa split grupal por subpartida.

Las transformaciones se ajustan con train y se aplican sobre test.


In [ ]:
# ============================================================================
# 2. SPLIT SIN LEAKAGE
# ============================================================================

def detectar_columna_temporal(df: pd.DataFrame) -> str | None:
    """
    Detecta una columna temporal candidata en el DataFrame.

    Args:
        df (pd.DataFrame): DataFrame de evaluacion.

    Returns:
        str | None: Nombre de columna temporal detectada o None.
    """
    candidatas = [
        'FECHA',
        'FEC_NUM',
        'FEC_ORDEN',
        'FEC_DECLARACION',
        'FECHA_DECLARACION',
        'ANIO_C'
    ]

    for columna in candidatas:
        if columna in df.columns:
            return columna

    return None


def crear_split_validacion(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Crea indices de train/test priorizando split temporal y luego split grupal.

    Args:
        df (pd.DataFrame): DataFrame de evaluacion.

    Returns:
        tuple[np.ndarray, np.ndarray, str]: indices train, indices test y descripcion del split.
    """
    columna_temporal = detectar_columna_temporal(df)

    if columna_temporal is not None:
        serie_tiempo = pd.to_datetime(df[columna_temporal], errors='coerce')

        if serie_tiempo.notna().mean() >= 0.7:
            corte = serie_tiempo.quantile(0.80)
            idx_train = df.index[serie_tiempo <= corte].to_numpy()
            idx_test = df.index[serie_tiempo > corte].to_numpy()

            if len(idx_train) > 0 and len(idx_test) > 0:
                return (
                    idx_train,
                    idx_test,
                    f'Split temporal 80/20 usando {columna_temporal}'
                )

        if pd.api.types.is_numeric_dtype(df[columna_temporal]):
            corte = df[columna_temporal].quantile(0.80)
            idx_train = df.index[df[columna_temporal] <= corte].to_numpy()
            idx_test = df.index[df[columna_temporal] > corte].to_numpy()

            if len(idx_train) > 0 and len(idx_test) > 0:
                return (
                    idx_train,
                    idx_test,
                    f'Split temporal/numerico 80/20 usando {columna_temporal}'
                )

    if 'ADUANA' in df.columns and df['ADUANA'].nunique() >= 2:
        grupos = df['ADUANA'].astype(str)
        splitter = GroupKFold(n_splits=min(5, grupos.nunique()))
        idx_train, idx_test = next(splitter.split(df, df['_gt'], groups=grupos))

        return (
            idx_train,
            idx_test,
            'GroupKFold usando ADUANA como grupo'
        )

    grupos = df['NUM_SPN_R'].astype(str)
    splitter = GroupKFold(n_splits=min(5, grupos.nunique()))
    idx_train, idx_test = next(splitter.split(df, df['_gt'], groups=grupos))

    return (
        idx_train,
        idx_test,
        'GroupKFold usando NUM_SPN_R como grupo'
    )


idx_train, idx_test, descripcion_split = crear_split_validacion(df_eval)

df_train = df_eval.iloc[idx_train].copy()
df_test = df_eval.iloc[idx_test].copy()

print('Split definido')
print(f'Metodo        : {descripcion_split}')
print(f'Train         : {len(df_train):,} registros')
print(f'Test          : {len(df_test):,} registros')
print(f'Outlier train : {df_train["_gt"].sum():,} ({100 * df_train["_gt"].mean():.2f}%)')
print(f'Outlier test  : {df_test["_gt"].sum():,} ({100 * df_test["_gt"].mean():.2f}%)')


## 4. Experimentos A/B leakage-safe

Cada variante cambia una sola cosa respecto al baseline:

| Variante | Cambio |
|---|---|
| A | Baseline: IQR sobre VU original |
| B | Transformacion: IQR sobre log1p(VU) |
| C | Preprocesamiento: IQR sobre VU winsorizado P5-P95 |
| D | Modelo: Isolation Forest sobre log1p(VU) |


In [ ]:
# ============================================================================
# 3. FUNCIONES LEAKAGE-SAFE PARA EXPERIMENTOS
# ============================================================================

def ajustar_estadisticos_iqr(
    df_train: pd.DataFrame,
    columna_valor: str,
    columna_grupo: str = 'NUM_SPN_R'
) -> pd.DataFrame:
    """
    Ajusta Q1, Q3 e IQR por grupo usando exclusivamente train.

    Args:
        df_train (pd.DataFrame): Datos de entrenamiento.
        columna_valor (str): Columna numerica a evaluar.
        columna_grupo (str): Columna de agrupacion.

    Returns:
        pd.DataFrame: Tabla de estadisticos por grupo.
    """
    stats = (
        df_train
        .groupby(columna_grupo)[columna_valor]
        .agg(
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75)
        )
        .reset_index()
    )

    stats['iqr'] = stats['q3'] - stats['q1']
    stats['limite_inf'] = stats['q1'] - 1.5 * stats['iqr']
    stats['limite_sup'] = stats['q3'] + 1.5 * stats['iqr']

    return stats


def predecir_iqr_con_stats(
    df_test: pd.DataFrame,
    stats: pd.DataFrame,
    columna_valor: str,
    columna_grupo: str = 'NUM_SPN_R'
) -> tuple[np.ndarray, np.ndarray]:
    """
    Predice outliers en test usando limites calculados en train.

    Args:
        df_test (pd.DataFrame): Datos de validacion.
        stats (pd.DataFrame): Estadisticos ajustados en train.
        columna_valor (str): Columna numerica evaluada.
        columna_grupo (str): Columna de agrupacion.

    Returns:
        tuple[np.ndarray, np.ndarray]: Prediccion binaria y score continuo.
    """
    tmp = df_test[[columna_grupo, columna_valor]].copy()

    tmp = tmp.merge(
        stats[[columna_grupo, 'limite_inf', 'limite_sup']],
        on=columna_grupo,
        how='left'
    )

    limite_inf_global = stats['limite_inf'].median()
    limite_sup_global = stats['limite_sup'].median()

    tmp['limite_inf'] = tmp['limite_inf'].fillna(limite_inf_global)
    tmp['limite_sup'] = tmp['limite_sup'].fillna(limite_sup_global)

    pred = (
        (tmp[columna_valor] < tmp['limite_inf']) |
        (tmp[columna_valor] > tmp['limite_sup'])
    ).astype(int).to_numpy()

    distancia_inf = (tmp['limite_inf'] - tmp[columna_valor]).clip(lower=0)
    distancia_sup = (tmp[columna_valor] - tmp['limite_sup']).clip(lower=0)
    score = (distancia_inf + distancia_sup).fillna(0).to_numpy()

    return pred, score


def winsorizar_test_con_train(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    columna_valor: str = 'VU',
    columna_grupo: str = 'NUM_SPN_R'
) -> pd.Series:
    """
    Aplica winsorizacion a test usando percentiles P5-P95 calculados en train.

    Args:
        df_train (pd.DataFrame): Datos de entrenamiento.
        df_test (pd.DataFrame): Datos de validacion.
        columna_valor (str): Columna numerica base.
        columna_grupo (str): Columna de agrupacion.

    Returns:
        pd.Series: Valores de test winsorizados.
    """
    percentiles = (
        df_train
        .groupby(columna_grupo)[columna_valor]
        .agg(
            p5=lambda x: x.quantile(0.05),
            p95=lambda x: x.quantile(0.95)
        )
        .reset_index()
    )

    tmp = df_test[[columna_grupo, columna_valor]].copy()
    tmp = tmp.merge(percentiles, on=columna_grupo, how='left')

    tmp['p5'] = tmp['p5'].fillna(percentiles['p5'].median())
    tmp['p95'] = tmp['p95'].fillna(percentiles['p95'].median())

    return tmp[columna_valor].clip(
        lower=tmp['p5'],
        upper=tmp['p95']
    )


def calcular_metricas_validacion(
    variante: str,
    modelo: str,
    cambio: str,
    y_true: np.ndarray,
    pred: np.ndarray,
    score: np.ndarray,
    latencia_ms: float
) -> dict:
    """
    Calcula metricas principales y secundarias para una variante.

    Args:
        variante (str): Codigo de variante.
        modelo (str): Nombre del metodo.
        cambio (str): Cambio aplicado respecto al baseline.
        y_true (np.ndarray): Etiqueta proxy de validacion.
        pred (np.ndarray): Prediccion binaria.
        score (np.ndarray): Score continuo para PR-AUC.
        latencia_ms (float): Tiempo de ejecucion en milisegundos.

    Returns:
        dict: Resultado resumido para tabla final.
    """
    return {
        'variante': variante,
        'modelo': modelo,
        'cambio': cambio,
        'f1': round(f1_score(y_true, pred, zero_division=0), 4),
        'precision': round(precision_score(y_true, pred, zero_division=0), 4),
        'recall': round(recall_score(y_true, pred, zero_division=0), 4),
        'pr_auc': round(average_precision_score(y_true, score), 4),
        'detectados': int(pred.sum()),
        'detectados_pct': round(100 * pred.mean(), 2),
        'latencia_ms': round(float(latencia_ms), 2)
    }


In [ ]:
# ============================================================================
# 4. EJECUCION DE EXPERIMENTOS SOBRE TEST
# ============================================================================

y_test = df_test['_gt'].to_numpy().astype(int)

resultados = []
scores_pr = {}

# A - Baseline
t0 = time.perf_counter()
stats_a = ajustar_estadisticos_iqr(df_train, 'VU')
pred_a_val, score_a_val = predecir_iqr_con_stats(df_test, stats_a, 'VU')
t_a = (time.perf_counter() - t0) * 1000
scores_pr['A - IQR original'] = score_a_val
resultados.append(
    calcular_metricas_validacion(
        variante='A',
        modelo='IQR original',
        cambio='Baseline - IQR sobre VU original',
        y_true=y_test,
        pred=pred_a_val,
        score=score_a_val,
        latencia_ms=t_a
    )
)

# B - log1p
t0 = time.perf_counter()
df_train_b = df_train.copy()
df_test_b = df_test.copy()
df_train_b['VU_LOG_SAFE'] = np.log1p(df_train_b['VU'])
df_test_b['VU_LOG_SAFE'] = np.log1p(df_test_b['VU'])
stats_b = ajustar_estadisticos_iqr(df_train_b, 'VU_LOG_SAFE')
pred_b_val, score_b_val = predecir_iqr_con_stats(df_test_b, stats_b, 'VU_LOG_SAFE')
t_b = (time.perf_counter() - t0) * 1000
scores_pr['B - IQR log1p'] = score_b_val
resultados.append(
    calcular_metricas_validacion(
        variante='B',
        modelo='IQR log1p',
        cambio='Var1 - transformacion log1p(VU)',
        y_true=y_test,
        pred=pred_b_val,
        score=score_b_val,
        latencia_ms=t_b
    )
)

# C - winsorizacion P5-P95 ajustada en train
t0 = time.perf_counter()
df_train_c = df_train.copy()
df_test_c = df_test.copy()
df_train_c['VU_WINS_SAFE'] = winsorizar_test_con_train(df_train, df_train)
df_test_c['VU_WINS_SAFE'] = winsorizar_test_con_train(df_train, df_test)
stats_c = ajustar_estadisticos_iqr(df_train_c, 'VU_WINS_SAFE')
pred_c_val, score_c_val = predecir_iqr_con_stats(df_test_c, stats_c, 'VU_WINS_SAFE')
t_c = (time.perf_counter() - t0) * 1000
scores_pr['C - IQR wins'] = score_c_val
resultados.append(
    calcular_metricas_validacion(
        variante='C',
        modelo='IQR winsorizado',
        cambio='Var2 - winsorizacion P5-P95 ajustada solo en train',
        y_true=y_test,
        pred=pred_c_val,
        score=score_c_val,
        latencia_ms=t_c
    )
)

# D - Isolation Forest entrenado solo con train
t0 = time.perf_counter()
modelo_if = IsolationForest(
    contamination=CONTAM,
    random_state=SEED
)
x_train = np.log1p(df_train[['VU']].to_numpy())
x_test = np.log1p(df_test[['VU']].to_numpy())
modelo_if.fit(x_train)
pred_if_raw = modelo_if.predict(x_test)
pred_d_val = (pred_if_raw == -1).astype(int)
score_d_val = -modelo_if.decision_function(x_test)
t_d = (time.perf_counter() - t0) * 1000
scores_pr['D - IF log1p'] = score_d_val
resultados.append(
    calcular_metricas_validacion(
        variante='D',
        modelo='Isolation Forest log1p',
        cambio='Var3 - modelo no supervisado entrenado solo con train',
        y_true=y_test,
        pred=pred_d_val,
        score=score_d_val,
        latencia_ms=t_d
    )
)

tabla_semana6 = (
    pd.DataFrame(resultados)
    .sort_values(['f1', 'precision', 'recall'], ascending=[False, False, False])
    .reset_index(drop=True)
)

baseline_f1_s6 = tabla_semana6.loc[tabla_semana6['variante'] == 'A', 'f1'].iloc[0]

tabla_semana6['mejora_vs_a'] = np.where(
    baseline_f1_s6 == 0,
    np.where(tabla_semana6['variante'] == 'A', '0.0%', 'N/A'),
    (((tabla_semana6['f1'] - baseline_f1_s6) / baseline_f1_s6) * 100).round(1).astype(str) + '%'
)

print('TABLA FINAL COMPARABLE - SEMANA 6')
print('=' * 120)
print(tabla_semana6.to_string(index=False))


## 5. Grafico unico para presentar

La guia pide un grafico unico. Este bloque genera una figura unica con PR curve y ranking F1.


In [ ]:
# ============================================================================
# 5. GRAFICO UNICO
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nombre, score in scores_pr.items():
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, score)
    pr_auc = average_precision_score(y_test, score)

    axes[0].step(
        recall_curve,
        precision_curve,
        where='post',
        linewidth=2,
        label=f'{nombre} | PR-AUC={pr_auc:.3f}'
    )

axes[0].axhline(
    y=y_test.mean(),
    linestyle='--',
    linewidth=1,
    label='Base rate'
)
axes[0].set_title('PR curve por variante')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8)

tabla_plot = tabla_semana6.sort_values('f1', ascending=True)
axes[1].barh(tabla_plot['modelo'], tabla_plot['f1'])
axes[1].set_title('Ranking por F1')
axes[1].set_xlabel('F1-score')
axes[1].set_xlim(0, 1)

for i, valor in enumerate(tabla_plot['f1']):
    axes[1].text(valor + 0.01, i, f'{valor:.4f}', va='center', fontsize=9)

fig.suptitle('Entrega semana 6 - Resultados comparables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=140, bbox_inches='tight')
plt.show()

print(f'Grafico guardado: {FIG_PATH}')


## 6. Reproducibilidad: comandos, logs, snapshot y hashes

Este bloque genera archivos de soporte con delimitador `|` y sin acentos en TXT/CSV.


In [ ]:
# ============================================================================
# 6. REPRODUCIBILIDAD
# ============================================================================

def calcular_hash_archivo(ruta: Path) -> str:
    """
    Calcula hash SHA256 de un archivo.

    Args:
        ruta (Path): Ruta del archivo.

    Returns:
        str: Hash SHA256 hexadecimal o cadena vacia si no existe.
    """
    if not ruta.exists():
        return ''

    sha = hashlib.sha256()

    with ruta.open('rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b''):
            sha.update(bloque)

    return sha.hexdigest()


tabla_semana6_export = tabla_semana6.copy()
tabla_semana6_export.to_csv(
    METRICAS_PATH,
    sep='|',
    index=False,
    encoding='utf-8'
)

snapshot = {
    'fecha_ejecucion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'seed': int(SEED),
    'contam': float(CONTAM),
    'metodo_split': descripcion_split,
    'registros_total': int(len(df_eval)),
    'registros_train': int(len(df_train)),
    'registros_test': int(len(df_test)),
    'outliers_test': int(df_test['_gt'].sum()),
    'archivo_metricas': str(METRICAS_PATH),
    'hash_metricas_sha256': calcular_hash_archivo(METRICAS_PATH),
    'archivo_figura': str(FIG_PATH),
    'hash_figura_sha256': calcular_hash_archivo(FIG_PATH)
}

SNAPSHOT_PATH.write_text(
    json.dumps(snapshot, indent=2, ensure_ascii=True),
    encoding='utf-8'
)

readme = f'''README REPRODUCIBILIDAD - SEMANA 6
============================================================

1. Ejecutar primero:
   jupyter notebook sprint2_fe_ab_testing(2).ipynb

2. Ejecutar despues:
   jupyter notebook sprint_semana6_pendientes.ipynb

3. Archivos generados:
   - {METRICAS_PATH}
   - {FIG_PATH}
   - {SNAPSHOT_PATH}
   - {README_PATH}

4. Configuracion:
   - seed|{SEED}
   - contamination|{CONTAM}
   - split|{descripcion_split}
   - train_registros|{len(df_train)}
   - test_registros|{len(df_test)}

5. Hashes:
   - metricas_sha256|{snapshot["hash_metricas_sha256"]}
   - figura_sha256|{snapshot["hash_figura_sha256"]}

6. Nota:
   La columna _gt es una etiqueta proxy estadistica para comparacion experimental.
   No reemplaza validacion experta.
'''

README_PATH.write_text(readme, encoding='utf-8')

print('Archivos de reproducibilidad generados')
print(f'Metricas : {METRICAS_PATH}')
print(f'Figura   : {FIG_PATH}')
print(f'Snapshot : {SNAPSHOT_PATH}')
print(f'README   : {README_PATH}')


## 7. Conclusion, decision, riesgos y siguiente sprint

Este bloque imprime el cierre ejecutivo listo para exponer.


In [ ]:
# ============================================================================
# 7. CIERRE EJECUTIVO
# ============================================================================

ganador = tabla_semana6.iloc[0]
baseline = tabla_semana6[tabla_semana6['variante'] == 'A'].iloc[0]

print('=' * 90)
print('CIERRE EJECUTIVO - ENTREGA SPRINT SEMANA 6')
print('=' * 90)

print('\n1. Contexto')
print(
    'Proyecto: deteccion de variaciones atipicas en valor unitario de exportaciones. '
    'Dataset: registros aduaneros filtrados y agrupados por subpartida. '
    'Metrica central: F1-score para la clase outlier.'
)

print('\n2. Baseline')
print(
    f'Baseline A: {baseline["modelo"]}. '
    f'F1={baseline["f1"]}, precision={baseline["precision"]}, '
    f'recall={baseline["recall"]}, PR-AUC={baseline["pr_auc"]}.'
)

print('\n3. Decision tecnica')
print(
    f'Se adopta la variante {ganador["variante"]} - {ganador["modelo"]}, '
    f'porque obtuvo el mayor F1 en validacion. '
    f'F1={ganador["f1"]}, mejora_vs_baseline={ganador["mejora_vs_a"]}, '
    f'latencia_ms={ganador["latencia_ms"]}.'
)

print('\n4. Justificacion')
print(
    'La decision prioriza equilibrio entre precision y recall, no solo cantidad de detectados. '
    'Ademas, el split evita que los estadisticos de test entren en el ajuste de las transformaciones.'
)

print('\n5. Riesgos')
print('- Riesgo 1: _gt es proxy estadistica, no etiqueta experta.')
print('- Riesgo 2: puede existir drift por cambio de subpartida, aduana o periodo.')

print('\n6. Proximo sprint')
print('- Validar una muestra de outliers con criterio experto y documentos comerciales.')
print('- Monitorear drift por subpartida/aduana y recalibrar contaminacion por grupo.')
print('=' * 90)
